# Day 03 — Functions + Data Structures (deep dive)
**Module 01: Python + Async + FastAPI for LLM Engineering**

> **What you'll learn:** def · parameters · return · default values · `*args` · `**kwargs` · functions as objects · `lambda` · `sorted()` · `map()` · `filter()` · list methods · dict methods · set operations · tuple + `zip()`

> **Why today matters:** In Day 02 you copy-pasted the same if/elif routing block every time you needed it. Today you wrap it in a function and write it **once**. Every LLM API call you build from Day 04 onwards uses functions built here.


---


## 1 — Functions

A function is a **named, reusable block of code**. Write it once — call it anywhere.

```python
def function_name(parameter1, parameter2):
    """One-line description."""
    # code here
    return result
```

| Keyword | Purpose |
|---------|--------|
| `def` | marks the start of a function definition |
| `( )` | parameters live here |
| `:` | required at the end of the `def` line |
| `return` | sends a value back to the caller |
| indentation | 4 spaces — the function body |


In [ ]:
# No parameters, no return — just executes
def greet():
    print("Hello! Welcome.")

greet()   # call once
greet()   # call again — same code, no copy-paste


In [ ]:
# Parameters + return value
def add(a, b):
    """Return the sum of a and b."""
    return a + b

print(add(3, 7))    # 10
print(add(10, 20))  # 30


### ShopSmart — route_query()

The `if/elif/else` routing block from Day 02 is now a function.  
Write it **once** → call it for any query, any number of times.


In [ ]:
def route_query(query: str) -> str:
    """Route a customer query to the correct support agent."""
    q = query.lower()
    if   "cancel" in q or "refund"   in q: return "human_agent"
    elif "track"  in q or "where"    in q: return "order_agent"
    elif "return" in q or "exchange" in q: return "returns_agent"
    elif "price"  in q or "discount" in q: return "promotions_agent"
    elif "product" in q or "spec"    in q: return "catalog_agent"
    else:                                   return "general_agent"

queries = [
    "Where is my order?",
    "I want a refund",
    "Do you have discounts?",
]

# ITERATION TRACE:
# Iter 1 → query="Where is my order?"  q="where is my order?"
#           "cancel"? No | "where"? Yes → return "order_agent"
#
# Iter 2 → query="I want a refund"     q="i want a refund"
#           "cancel"? No | "refund"? Yes → return "human_agent"
#
# Iter 3 → query="Do you have discounts?"  q="do you have discounts?"
#           "cancel"? No | "where"? No | "discount"? Yes → return "promotions_agent"

for q in queries:
    print(f"  '{q}'  →  {route_query(q)}")


---


## 2 — Default Parameter Values

A parameter can have a default value.  
If the caller does not pass it → default is used.  
If the caller does pass it → caller's value wins.

> **Rule:** parameters WITH defaults must come AFTER parameters WITHOUT defaults.
```python
def fn(required, optional=default)   # correct
def fn(optional=default, required)   # SyntaxError
```


In [6]:
base = 5
exponent = 3
base ** exponent

125

In [10]:
def power(base,exponent=2):
    return base ** exponent

power(3,4)

81

In [ ]:
def power(base, exponent=2):
    """Return base raised to exponent. Default: squared."""
    return base ** exponent

print(power(3))       # 3**2 = 9   — uses default exponent=2
print(power(3, 3))    # 3**3 = 27  — caller overrides to 3
print(power(2, 10))   # 2**10 = 1024


In [ ]:
# ShopSmart — max_sentences has a sensible default for most callers
# Callers that need different limits can override explicitly
def build_system_prompt(role: str, domain: str, max_sentences: int = 3) -> str:
    """Build a system prompt following Module 00 Technique 01 template."""
    return (
        f"You are a {role} for {domain}. "
        f"Respond in {max_sentences} sentences maximum. "
        f"Only use information provided — never invent details."
    )

p1 = build_system_prompt("support agent", "ShopSmart")           # uses default 3
p2 = build_system_prompt("support agent", "ShopSmart", max_sentences=1)  # override

print("Default (3):", p1)
print()
print("Override (1):", p2)


---


## 3 — `*args` — Variable-Length Positional Arguments

`*args` collects **any number of positional arguments** into a **tuple** inside the function.

```python
def fn(*args):
    # args is a tuple of all positional values passed in
    for item in args:
        ...
```

Use `*args` when the caller might pass one value or ten — without changing the function signature.


In [14]:
def total(*numbers):
    """Add any number of values."""
    result = 0
    for n in numbers:
        result += n # result = result + n
    return result

# numbers is a tuple inside the function:
# total(1, 2)       → numbers = (1, 2)
# total(1,2,3,4,5)  → numbers = (1, 2, 3, 4, 5)

print(total(1, 2))             # 3
print(total(1, 2, 3, 4, 5))   # 15
print(total(10, 20, 30))       # 60


3
15
60


In [15]:
def aFunction(a,*b):
    print(a)
    print(b)

aFunction(1,2,3,4,5)

1
(2, 3, 4, 5)


In [19]:
aString = "abcde\n\nfghik\tkkkk"
print(aString)

abcde

fghik	kkkk


In [33]:
aString = " "
if aString.strip():
    print(aString)

In [37]:
seperator = "\n\n"
aStringList = [' Hello How are you?  ',"","Everything alright"]
# for aString in aStringList:
#     tmp = aString.strip()
#     print("original:",f"$${aString}$$", "After striping:",f"$${type(tmp)}$$")

seperator.join(aStringList)

' Hello How are you?  \n\n\n\nEverything alright'

In [38]:
# ShopSmart — combine_context() accepts however many RAG chunks
# the retrieval step returns. Caller never needs to count them.
def combine_context(*chunks: str, separator: str = "\n\n") -> str:
    """
    Join multiple RAG context chunks into one string.
    Empty chunks are silently ignored.
    separator= is keyword-only (comes after *chunks).
    """
    # TRACE: chunks = ('[Doc 1]...', '[Doc 2]...', '')
    # Iter 1 → chunk='[Doc 1]...'  strip() non-empty → include
    # Iter 2 → chunk='[Doc 2]...'  strip() non-empty → include
    # Iter 3 → chunk=''            strip() = '' → skip (falsy)
    # non_empty = ['[Doc 1]...', '[Doc 2]...']
    # result = '[Doc 1]...\n\n[Doc 2]...'
    non_empty = [c.strip() for c in chunks if c.strip()]
    return separator.join(non_empty)

ctx = combine_context(
    "[Doc 1] Classic Monitor: 27-inch 4K display, $205.21.",
    "[Doc 2] Return policy: 30 days from delivery.",
    "",   # empty chunk — ignored automatically
)
print(ctx)
print(f"\nTotal chars: {len(ctx)}")


[Doc 1] Classic Monitor: 27-inch 4K display, $205.21.

[Doc 2] Return policy: 30 days from delivery.

Total chars: 100


---


## 4 — `**kwargs` — Variable-Length Keyword Arguments

`**kwargs` collects **any number of keyword arguments** into a **dict** inside the function.

```python
def fn(**kwargs):
    # kwargs is a dict of all key=value pairs passed in
    for key, value in kwargs.items():
        ...
```

Use `**kwargs` when callers may pass optional named parameters you cannot predict in advance — like LLM API options.


In [40]:
def describe(**details):
    """Print whatever keyword arguments are passed in."""
    # details = {'name': 'Alice', 'age': 25, 'city': 'Hyderabad'}
    print(details)
    print(details.items())
    for key, value in details.items():
        print(f"  {key}: {value}")

describe(name="Alice", age=25, city="Hyderabad")


{'name': 'Alice', 'age': 25, 'city': 'Hyderabad'}
dict_items([('name', 'Alice'), ('age', 25), ('city', 'Hyderabad')])
  name: Alice
  age: 25
  city: Hyderabad


In [42]:
def aFunction(a,*b,**c):
    print(a)
    print(b)
    print(c)

aFunction(1,2,3,4,5,e=1,f=2,g=3)

1
(2, 3, 4, 5)
{'e': 1, 'f': 2, 'g': 3}


In [44]:
aDict = {"model": "gpt-4o", "messages": [{"role": "user", "content": "Hello"}]}
aDict2 = {"temp":0.2,"max_tokens":512,"stream":True}
aDict.update(aDict2)
print(aDict)

{'model': 'gpt-4o', 'messages': [{'role': 'user', 'content': 'Hello'}], 'temp': 0.2, 'max_tokens': 512, 'stream': True}


In [46]:
from typing import Any

# ShopSmart — build_api_request() always needs model and messages.
# Everything else (temperature, max_tokens, stream...) is optional
# and changes per call. **kwargs handles all of them.
def build_api_request(model: str, messages: list, **kwargs: Any) -> dict:
    """
    Build an OpenAI-compatible API request payload.
    Required: model, messages.
    Optional via **kwargs: temperature, max_tokens, stream, top_p, etc.
    """
    payload = {"model": model, "messages": messages}
    print(payload)
    print(kwargs)
    payload.update(kwargs)   # merge all optional params
    print(payload)
    return payload

# kwargs = {'temperature': 0.2, 'max_tokens': 512, 'stream': True}
payload = build_api_request(
    model       = "gpt-4o",
    messages    = [{"role": "user", "content": "Hello"}],
    temperature = 0.2,
    max_tokens  = 512,
    stream      = True,
)

for k, v in payload.items():
    if k != "messages":
        print(f"  {k}: {v}")


{'model': 'gpt-4o', 'messages': [{'role': 'user', 'content': 'Hello'}]}
{'temperature': 0.2, 'max_tokens': 512, 'stream': True}
{'model': 'gpt-4o', 'messages': [{'role': 'user', 'content': 'Hello'}], 'temperature': 0.2, 'max_tokens': 512, 'stream': True}
  model: gpt-4o
  temperature: 0.2
  max_tokens: 512
  stream: True


---


## 5 — Functions as Objects

Functions in Python are **first-class objects** — you can assign them, pass them, and return them.

| Operation | Syntax | Note |
|-----------|--------|------|
| Assign to variable | `op = square` | No `()` — assigns the function itself |
| Call the assigned variable | `op(5)` | Adds `()` to call it |
| Pass into another function | `apply(square, 4)` | The receiving function calls it |

> `square` = the function object  
> `square()` = call the function, get back its return value


In [53]:
def square(n): return n * n
def cube(n):   return n * n * n

# print(square(2))
# print(cube(2))

# higher order function
def abc(function , value):
    print(function,value)
    return function(value)

print(abc(square, 2))
print(abc(cube, 3))



<function square at 0x10c196480> 2
4
<function cube at 0x10c196340> 3
27


In [ ]:
def square(n): return n * n
def cube(n):   return n * n * n

# # Assign function to variable — no () means assign the object, not call it
# operation = square
# print(f"operation = square  →  operation(5) = {operation(5)}")  # 25
#
# operation = cube
# print(f"operation = cube    →  operation(5) = {operation(5)}")  # 125

# Pass a function as an argument
def apply(fn, value):
    """Call fn with value and return the result."""
    return fn(value)

print(f"apply(square, 4) = {apply(square, 4)}")  # 16
print(f"apply(cube,   4) = {apply(cube,   4)}")  # 64


---


## 6 — lambda · sorted() · map() · filter()

`lambda` is a **one-line anonymous function**. Used almost exclusively with `sorted()`, `map()`, `filter()`.

```python
lambda x: x["price"]      # for each x, return x["price"]
# equivalent to:
def get_price(x):
    return x["price"]
```

| Function | What it does | Returns |
|----------|-------------|--------|
| `sorted(items, key=fn)` | sort by whatever `fn` returns | new sorted list |
| `map(fn, items)` | apply `fn` to every item | iterator (wrap in `list()`) |
| `filter(fn, items)` | keep items where `fn` returns `True` | iterator (wrap in `list()`) |


In [11]:
product = {"name": "Classic Monitor",   "price": 205.21, "rating": 4.2}
get_price = lambda x: x["price"]
price = get_price(product)
print(price)

205.21


In [ ]:
prices = [205.21, 568.17, 45.00, 29.99]
print(sorted(prices))

In [57]:
products = [
    {"name": "Classic Monitor",   "price": 205.21, "rating": 4.2},
    {"name": "Ultimate Perfume",  "price": 568.17, "rating": 3.8},
    {"name": "Yoga Mat Pro",      "price":  45.00, "rating": 4.7},
    {"name": "Budget Headphones", "price":  29.99, "rating": 3.1},
]

# sorted() with key= lambda
# TRACE key extraction: 205.21, 568.17, 45.00, 29.99  → sorted asc

by_price  = sorted(products, key=lambda x: x["price"])

# TRACE key extraction: 4.2, 3.8, 4.7, 3.1  → sorted desc
by_rating = sorted(products, key=lambda x: x["rating"], reverse=True)

print("Cheapest first:")
for p in by_price:
    # Format the price as a floating-point number:
    # 7  -> reserve a minimum width of 7 characters for alignment
    # .2 -> display exactly 2 digits after the decimal point
    # f  -> use fixed-point decimal notation
    # This ensures prices line up neatly in tabular output.
    print(f"  ${p['price']:7.2f}  {p['name']}")

print("\nHighest rated first:")
for p in by_rating:
    print(f"  ★{p['rating']}  {p['name']}")


Cheapest first:
  $  29.99  Budget Headphones
  $  45.00  Yoga Mat Pro
  $ 205.21  Classic Monitor
  $ 568.17  Ultimate Perfume

Highest rated first:
  ★4.7  Yoga Mat Pro
  ★4.2  Classic Monitor
  ★3.8  Ultimate Perfume
  ★3.1  Budget Headphones


[29.99, 45.0, 205.21, 568.17]


In [1]:
prices = [205.21, 568.17, 45.00, 29.99]

def rounded(p: float) -> float:
    return round(p * 0.9, 2)

# ── map() ──────────────────────────────────────────────────
# map(lambda p: round(p * 0.9, 2), prices)
# lambda takes each price p, keeps 90% of it, rounds to 2 decimal places
#
# HOW map WORKS:
#   Takes the lambda function AND the iterator (prices list) as inputs
#   Calls the lambda once per item — passes the item as the argument
#   Collects every return value into a new iterator
#
# ITERATION TRACE:
# Iter 1 → p = 205.21  →  round(205.21 * 0.9, 2)  →  round(184.689, 2)  →  184.69
# Iter 2 → p = 568.17  →  round(568.17 * 0.9, 2)  →  round(511.353, 2)  →  511.35
# Iter 3 → p =  45.00  →  round(45.00  * 0.9, 2)  →  round(40.5,   2)   →   40.5
# Iter 4 → p =  29.99  →  round(29.99  * 0.9, 2)  →  round(26.991, 2)   →   26.99
# Output iterator: [184.69, 511.35, 40.5, 26.99]
#
# map() returns an iterator — wrap in list() to get a regular Python list
discounted = list(map(lambda p: rounded(p), prices))


# ── filter() ───────────────────────────────────────────────
# filter(lambda p: p < 100, prices)
# lambda must return True or False for every item
#   True  → keep the item in the output
#   False → skip it, do NOT include in output
#
# HOW filter WORKS:
#   Argument 1: lambda — must return True or False
#   Argument 2: iterator (list, tuple, dict, set, etc.)
#   Only items where lambda returns True make it to the output
#
# ITERATION TRACE:
# Iter 1 → p = 205.21  →  205.21 < 100  →  False  →  skip ✗
# Iter 2 → p = 568.17  →  568.17 < 100  →  False  →  skip ✗
# Iter 3 → p =  45.00  →   45.00 < 100  →  True   →  keep ✓  →  45.00
# Iter 4 → p =  29.99  →   29.99 < 100  →  True   →  keep ✓  →  29.99
# Output iterator: [45.00, 29.99]
#
# filter() also returns an iterator — wrap in list()
affordable = list(filter(lambda p: p < 100, prices))


print(f"Original  : {prices}")
print(f"10% off   : {discounted}")
print(f"Under $100: {affordable}")


Original  : [205.21, 568.17, 45.0, 29.99]
10% off   : [184.69, 511.35, 40.5, 26.99]
Under $100: [45.0, 29.99]


---


## 7 — List Methods (deep dive)

| Method | What it does | Returns |
|--------|-------------|--------|
| `.append(x)` | Add one item to the end | `None` (modifies in place) |
| `.extend([x,y])` | Add multiple items to the end | `None` |
| `.pop()` | Remove and return last item | the removed item |
| `.pop(i)` | Remove and return item at index i | the removed item |
| `.sort()` | Sort in place ascending | `None` |
| `.sort(reverse=True)` | Sort in place descending | `None` |
| `.reverse()` | Reverse in place | `None` |
| `.index(x)` | First index of value x | `int` |
| `.count(x)` | How many times x appears | `int` |
| `.insert(i, x)` | Insert x at index i | `None` |
| `.remove(x)` | Remove first occurrence of x | `None` |
| `.clear()` | Remove all items | `None` |
| `list[a:b]` | Slice from a to b-1 | new list |


In [ ]:
nums = [3, 1, 4, 1, 5, 9]

print(f"Start         : {nums}")

nums.append(2)
print(f"append(2)     : {nums}")

nums.extend([6, 5])
print(f"extend([6,5]) : {nums}")

popped = nums.pop()
print(f"pop()         : {nums}  ← removed: {popped}")

nums.sort()
print(f"sort()        : {nums}")

nums.reverse()
print(f"reverse()     : {nums}")

print(f"index(9)      : {nums.index(9)}")
print(f"count(1)      : {nums.count(1)}")
print(f"slice [1:4]   : {nums[1:4]}")


### ShopSmart — LLM message history

The conversation history sent to the OpenAI API is a **list of dicts** that grows one `.append()` at a time.  
This is the exact pattern used in every LLM chat application.


In [ ]:
system_prompt = build_system_prompt("support agent", "ShopSmart")
messages: list = [{"role": "system", "content": system_prompt}]

# ITERATION TRACE — list grows with each append:
# After append 1 → messages has 2 items [{system}, {user:Where is...}]
# After append 2 → messages has 3 items [..., {assistant:Could you...}]
# After append 3 → messages has 4 items [..., {user:It is john21...}]

messages.append({"role": "user",      "content": "Where is my order #3042?"})
messages.append({"role": "assistant", "content": "Could you confirm your email?"})
messages.append({"role": "user",      "content": "It is john21@example.net"})

for msg in messages:
    print(f"{msg['role']:12s} | {msg['content']}")

print(f"\nTotal turns: {len(messages)}")


---


## 8 — Dict Methods (deep dive)

| Method | What it does | Returns |
|--------|-------------|--------|
| `d[key]` | Access value — **raises KeyError if missing** | value |
| `.get(key)` | Access value — returns `None` if missing | value or None |
| `.get(key, default)` | Access value — returns default if missing | value or default |
| `.keys()` | All keys | view |
| `.values()` | All values | view |
| `.items()` | All (key, value) pairs | view |
| `.update({k:v})` | Add/overwrite key-value pairs | `None` |
| `.pop(key)` | Remove key and return its value | value |
| `{**d1, **d2}` | Merge two dicts (d2 wins on duplicates) | new dict |


In [ ]:
person = {"name": "Alice", "age": 25}

print(".keys()  :", list(person.keys()))
print(".values():", list(person.values()))
print(".items() :", list(person.items()))

print(".get('name')        :", person.get("name"))
print(".get('phone','N/A') :", person.get("phone", "N/A"))  # safe — no KeyError

person.update({"city": "Hyderabad", "age": 26})
print(".update(...):", person)

removed = person.pop("city")
print(".pop('city'):", person, "← removed:", removed)


In [ ]:
# .get() is critical when reading LLM JSON responses
# The LLM was told to return specific fields — it doesn't always comply.
# d["missing_key"]            → KeyError — CRASHES the service
# d.get("missing_key", "N/A") → "N/A"   — safe fallback

llm_response = {"category": "TRACK_ORDER", "confidence": "high"}
# note: "reason" key is missing from this response

category   = llm_response.get("category",   "UNKNOWN")
confidence = llm_response.get("confidence", "low")
reason     = llm_response.get("reason",     "not provided")  # missing key

print(f"category  : {category}")
print(f"confidence: {confidence}")
print(f"reason    : {reason}   ← key missing, default returned")


In [ ]:
# .items() — loop over key-value pairs together
# ITERATION TRACE:
# Iter 1 → key="model",       value="gpt-4o"
# Iter 2 → key="max_tokens",  value=1024
# Iter 3 → key="temperature", value=0.2
# Iter 4 → key="stream",      value=False
model_config = {"model": "gpt-4o", "max_tokens": 1024, "temperature": 0.2, "stream": False}

for key, value in model_config.items():
    print(f"  {key:15s}: {value}")


In [ ]:
# Dict unpacking with ** — merge two dicts
# Keys from the SECOND dict override keys from the first
#
# TRACE: defaults  = {temperature:0.2, max_tokens:512, stream:False}
#        overrides = {temperature:0.0, max_tokens:256}
#        merge step:
#          temperature → 0.2 from defaults, then OVERWRITTEN by 0.0 from overrides
#          max_tokens  → 512 from defaults, then OVERWRITTEN by 256 from overrides
#          stream      → False from defaults, no override → stays False
#        merged = {temperature:0.0, max_tokens:256, stream:False}

defaults  = {"temperature": 0.2, "max_tokens": 512, "stream": False}
overrides = {"temperature": 0.0, "max_tokens": 256}
merged    = {**defaults, **overrides}

print("defaults :", defaults)
print("overrides:", overrides)
print("merged   :", merged, "← overrides win on duplicate keys")


---


## 9 — Set Operations (deep dive)

| Operation | Syntax | What it does |
|-----------|--------|--------------|
| Add item | `.add(x)` | Add x (no-op if already present) |
| Remove safely | `.discard(x)` | Remove x — no error if missing |
| Remove (strict) | `.remove(x)` | Remove x — **KeyError** if missing |
| Union | `a \| b` | All items from both sets |
| Intersection | `a & b` | Items present in both sets |
| Difference | `a - b` | Items in a but not in b |
| Sym. difference | `a ^ b` | Items in one but not both |
| Subset | `a.issubset(b)` | Is every item of a also in b? |
| Superset | `a.issuperset(b)` | Does a contain all items of b? |
| Membership | `x in s` | O(1) — instant regardless of size |


In [ ]:
a = {1, 2, 3, 4}
b = {3, 4, 5, 6}

print(f"a = {a}")
print(f"b = {b}")
print(f"a | b  (union)        : {a | b}")
print(f"a & b  (intersection) : {a & b}")
print(f"a - b  (difference)   : {a - b}")
print(f"a ^ b  (sym diff)     : {a ^ b}")

a.add(10)
print(f"a.add(10)    : {a}")

a.discard(99)   # safe — no error if 99 is not there
print(f"a.discard(99): {a}  ← 99 not in set, no error")

print({1,2}.issubset(a))    # True — {1,2} is contained in a
print(a.issuperset({1,2}))  # True — a contains all of {1,2}


---


## 10 — Tuple + zip()

### Tuple — ordered, immutable
Use tuples when values **must not change**: config pairs, coordinate pairs, version records, function return values.

### zip() — pair two lists by position
```python
for a, b in zip(list1, list2):
    ...  # a and b are matched by index
```
Stops automatically when the shorter list is exhausted.


In [4]:
# Function returning multiple values — Python returns a tuple
def min_max(numbers):
    """Return (minimum, maximum) as a tuple."""
    min_value = min(numbers)
    max_value = max(numbers)
    return min_value,max_value   # Python packs into tuple automatically

result     = min_max([3, 1, 7, 2, 9])
print(type(result), result)            # <class 'tuple'>  (1, 9)

low, high  = min_max([3, 1, 7, 2, 9]) # unpack immediately
print(f"low={low}, high={high}")


<class 'tuple'> (1, 9)
low=1, high=9


In [2]:
aTuple = 1,2
print(aTuple)

(1, 2)


In [11]:
# zip() — pair two lists by position
names  = ["Alice", "Bob", "Carol"]
scores = [90, 75, 88]
print(list(zip(names, scores)))

# ITERATION TRACE:
# Iter 1 → name="Alice", score=90  → print "Alice: 90"
# Iter 2 → name="Bob",   score=75  → print "Bob: 75"
# Iter 3 → name="Carol", score=88  → print "Carol: 88"
# shorter list exhausted → loop ends

for name, score in zip(names, scores):
    print(f"  {name}: {score}")


[('Alice', 90, 4)]
  Alice: 90
  Bob: 75
  Carol: 88


In [12]:
# ShopSmart — build conversation history from two parallel lists
user_messages = ["Where is my order?",      "What is the return policy?"]
asst_messages = ["Please share your order ID.","Returns accepted within 30 days."]

# ITERATION TRACE:
# Iter 1 → user_msg="Where is my order?", asst_msg="Please share your order ID."
#           append {role:user, content:"Where is..."}
#           append {role:assistant, content:"Please share..."}
#           conversation has 2 items
#
# Iter 2 → user_msg="What is the return policy?", asst_msg="Returns accepted..."
#           append {role:user, content:"What is..."}
#           append {role:assistant, content:"Returns accepted..."}
#           conversation has 4 items

conversation = []
for user_msg, asst_msg in zip(user_messages, asst_messages):
    conversation.append({"role": "user",      "content": user_msg})
    conversation.append({"role": "assistant", "content": asst_msg})

print(conversation)
for msg in conversation:
    print(f"  {msg['role']:12s} | {msg['content']}")


[{'role': 'user', 'content': 'Where is my order?'}, {'role': 'assistant', 'content': 'Please share your order ID.'}, {'role': 'user', 'content': 'What is the return policy?'}, {'role': 'assistant', 'content': 'Returns accepted within 30 days.'}]
  user         | Where is my order?
  assistant    | Please share your order ID.
  user         | What is the return policy?
  assistant    | Returns accepted within 30 days.


In [ ]:
# Prompt version history — list of tuples
# Tuples prevent accidental modification of the records
prompt_versions = [
    (1, "Initial ShopSmart support prompt"),
    (2, "Added refund handling rules"),
    (3, "Improved output format — Technique 04"),
    (4, "Added few-shot examples — Technique 02"),
]

# ITERATION TRACE:
# Iter 1 → version=1, description="Initial ShopSmart..."
# Iter 2 → version=2, description="Added refund..."
# Iter 3 → version=3, description="Improved output..."
# Iter 4 → version=4, description="Added few-shot..."

for version, description in prompt_versions:
    print(f"  v{version}: {description}")

latest_v, latest_d = prompt_versions[-1]   # unpack last tuple
print(f"\nLatest: v{latest_v} — {latest_d}")


---


## Summary — Day 03

| Concept | Syntax / Rule |
|---------|---------------|
| `def` | `def fn(p1, p2): ... return result` — write once, call anywhere |
| Default params | `def fn(required, optional=default)` — defaults come AFTER required |
| `*args` | Collects any number of positional args → **tuple** inside fn |
| `**kwargs` | Collects any number of keyword args → **dict** inside fn |
| Functions as objects | `op = square` assigns function · `op(5)` calls it |
| `lambda` | `lambda x: x["price"]` — one-line anonymous function |
| `sorted(items, key=fn)` | Sort by whatever `fn` returns |
| `map(fn, items)` | Apply `fn` to every item → wrap in `list()` |
| `filter(fn, items)` | Keep items where `fn` returns `True` → wrap in `list()` |
| List `.append(x)` | Add one item to end (modifies in place) |
| List `.extend([x,y])` | Add multiple items to end |
| List `.pop()` | Remove and return last item |
| Dict `.get(k, default)` | Safe access — never raises KeyError |
| Dict `.items()` | Loop over key-value pairs together |
| Dict `{**d1, **d2}` | Merge dicts — d2 keys override d1 keys |
| Set `in` | O(1) membership check — instant regardless of size |
| Set `\|` `&` `-` `^` | union · intersection · difference · symmetric diff |
| Tuple unpacking | `a, b = (x, y)` — assign each value in one line |
| `zip(list1, list2)` | Pair items by index · stops at shorter list |

**Connection to Module 00:**
- `build_system_prompt()` → Module 00 Technique 01 prompt anatomy in Python
- `build_api_request()` → assembles the full API payload the LLM receives
- `route_query()` → Module 00 routing rules, now reusable
- `combine_context()` → builds the `<context>` block from Technique 05 (RAG)

**Next:** Day 04 — File I/O + JSON + String Operations  
Loading system prompts from `.txt` files · parsing LLM JSON responses · string methods
